In [ ]:
# ---------- 0) 설치 ----------
#  - gcsfs==2025.3.0 이 fsspec==2025.3.0 을 요구하므로 버전을 맞춰 설치
#  - 그 뒤 NLP 스택을 설치(또는 업그레이드)합니다.
!pip -q install -U "fsspec==2025.3.0" "gcsfs==2025.3.0"
!pip -q install -U "transformers==4.44.2" \
                   "tokenizers==0.19.1" \
                   "datasets==2.20.0" \
                   "accelerate==0.34.2" \
                   scikit-learn pandas openpyxl jieba

In [ ]:
# ---------- 1) 임포트 & 기본 설정 ----------
import os, re, random, json
import numpy as np
import pandas as pd
from typing import List, Tuple

import jieba
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

BASE_MODEL = "IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment"  # 중국어 감성 모델(이진: [neg, pos])
OUTPUT_DIR = "/content/erlangshen_fullfinetune_sile_fix"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ---------- 2) 데이터 업로드 ----------
from google.colab import files
print("엑셀 파일을 업로드하세요")
uploaded = files.upload()
xlsx_path = list(uploaded.keys())[0]
print("Loaded:", xlsx_path)

df = pd.read_excel(xlsx_path)
print("원본 컬럼:", df.columns.tolist())
print(df.head(3))

In [ ]:
# ---------- 3) 컬럼 정규화 ----------
def find_first_col(candidates: List[str], columns: List[str]):
    lower_cols = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_cols:
            return lower_cols[cand.lower()]
    return None

TEXT_COL  = find_first_col(["text","sentence","문장"], df.columns.tolist())
LABEL_COL = find_first_col(["label","labels","라벨","정답"], df.columns.tolist())
DOMAIN_COL = find_first_col(["domain","category","카테고리","분류"], df.columns.tolist())

if TEXT_COL is None:
    raise ValueError("문장 컬럼(text/sentence/문장)이 필요합니다.")

def normalize_label(v):
    if isinstance(v, str):
        v = v.strip().lower()
        if v in ["pos","positive","긍정","1","true","yes","y"]:
            return 1
        if v in ["neg","negative","부정","0","false","no","n"]:
            return 0
        try:
            f = float(v)
            return 1 if f >= 0.5 else 0
        except:
            return 0
    if isinstance(v, (int, np.integer)):
        return 1 if v >= 1 else 0
    if isinstance(v, float):
        return 1 if v >= 0.5 else 0
    return 0

if LABEL_COL is None:
    print("라벨 컬럼이 없어 임시 0(부정)으로 채웁니다. 규칙 재라벨링에서 수정됩니다.")
    df["label"] = 0
    LABEL_COL = "label"

df[LABEL_COL] = df[LABEL_COL].apply(normalize_label)
print("정규화 후 라벨 분포:", df[LABEL_COL].value_counts(dropna=False).to_dict())

In [ ]:
# ---------- 4) 편향 교정 규칙 ----------
POS_ADJS_FOR_SILE = set("可爱 开心 激动 幸福 高兴 好听 乖 好吃 舒服 美 欢乐 漂亮".split())

FOOD_HINTS = set("""
饭 菜 面 包 子 馒头 馄饨 粥 饺 子 汤 甜点 糖 果 巧 克 力 巧克力 冰淇淋 雪糕 蛋糕 饮料 可乐 雪碧 奶茶
鱼 肉 牛肉 羊肉 猪肉 鸡肉 鸭肉 海鲜 虾 螃蟹 螺 蚝
水果 苹果 香蕉 橙 芒果 草莓 葡萄 西瓜 哈密瓜 桃 梨 菠萝 木瓜
""".split())

def has_any(token_list: List[str], targets: set) -> bool:
    return any(tok in targets for tok in token_list)

def tokenize_cjk(s: str) -> List[str]:
    return [w.strip() for w in jieba.cut(s) if w.strip()]

def detect_food_domain(s: str) -> str:
    toks = tokenize_cjk(s)
    return "FOOD" if has_any(toks, FOOD_HINTS) else "BEHAVIOR"

def has_adj_plus_sile(s: str) -> Tuple[bool, str]:
    if "死了" not in s:
        return (False, "")
    toks = tokenize_cjk(s)
    for t in toks:
        if t in POS_ADJS_FOR_SILE:
            return (True, t)
    for adj in POS_ADJS_FOR_SILE:
        if adj + "死了" in s or adj + " 死了" in s:
            return (True, adj)
    return (False, "")

def conditional_relabel(row):
    text = str(row[TEXT_COL])
    orig = int(row[LABEL_COL])
    corrected = orig

    # A) “死了” + 지정 형용사 → 긍정(1)
    has_sile, _ = has_adj_plus_sile(text)
    if has_sile:
        return 1

    # B) '甜' 조건부 (BEHAVIOR=1, FOOD=0)
    if "甜" in text:
        if DOMAIN_COL and pd.notna(row.get(DOMAIN_COL, None)):
            dom = str(row[DOMAIN_COL]).strip().upper()
        else:
            dom = detect_food_domain(text)
        if dom == "BEHAVIOR":
            corrected = 1
        elif dom == "FOOD":
            corrected = 0
    return corrected

df["corrected_label"] = df.apply(conditional_relabel, axis=1)
print("재라벨링 요약:\n", pd.crosstab(df[LABEL_COL], df["corrected_label"], rownames=["orig"], colnames=["corrected"]))

In [ ]:
# ---------- 5) (선택) 소량 보강 ----------
AUGMENT_MAX_MULTIPLIER = 3
def needs_sile_positive_boost(row):
    s = str(row[TEXT_COL])
    has_sile, _ = has_adj_plus_sile(s)
    return has_sile and row["corrected_label"] == 1

pos_sile_df = df[df.apply(needs_sile_positive_boost, axis=1)]
if len(pos_sile_df) > 0:
    aug_times = min(AUGMENT_MAX_MULTIPLIER-1, 2)
    aug_list = [pos_sile_df.copy() for _ in range(aug_times)]
    aug_df = pd.concat(aug_list, ignore_index=True) if aug_list else pd.DataFrame(columns=df.columns)
    df_aug = pd.concat([df, aug_df], ignore_index=True)
    print(f"보강 전:{len(df)}, 보강 후:{len(df_aug)} (+‘死了’ 긍정 복제)")
else:
    df_aug = df.copy()
    print("‘死了’ 긍정 보강 없음 → 생략")

df_aug["label_final"] = df_aug["corrected_label"].astype(int)

In [ ]:
# ---------- 6) 데이터 분할 ----------
train_df, test_df = train_test_split(
    df_aug[[TEXT_COL, "label_final"]].dropna(),
    test_size=0.1,
    random_state=SEED,
    stratify=df_aug["label_final"]
)
print("train:", train_df.shape, "test:", test_df.shape)

In [ ]:
# ---------- 7) 토크나이저/데이터셋 ----------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)

class SentDataset(Dataset):
    def __init__(self, df, text_col, label_col):
        self.texts = df[text_col].astype(str).tolist()
        self.labels = df[label_col].astype(int).tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return {"text": self.texts[idx], "labels": self.labels[idx]}

def preprocess(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding=False
    )
    enc["labels"] = batch["labels"]
    return enc

from datasets import Dataset as HFDataset
train_hf = HFDataset.from_list([{"text": t, "labels": l} for t, l in zip(train_df[TEXT_COL].astype(str), train_df["label_final"].astype(int))]).map(preprocess, batched=True)
test_hf  = HFDataset.from_list([{"text": t, "labels": l} for t, l in zip(test_df[TEXT_COL].astype(str),  test_df["label_final"].astype(int))]).map(preprocess, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# ---------- 8) 베이스 모델(학습 전) 로드 & 점검 ----------
def load_base_model():
    m = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,
        torch_dtype=torch.float32
    ).to(device)
    m.eval()
    return m

def predict_proba(model, texts: List[str]) -> List[Tuple[float,float]]:
    toks = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**toks).logits
        probs = torch.softmax(out, dim=-1).cpu().numpy()
    return [tuple(p) for p in probs]

base_model_eval = load_base_model()

probe_samples = [
    # ‘死了’ + 지정 형용사 → 목표: 긍정↑
    "可爱死了！", "开心死了！", "快乐死了！", "高兴死了！", "漂亮死了！", "帅气死了！", "好吃死了！",
    "厉害死了！", "幸福死了！", "她今天真是美丽死了！",
    # 甜의 맥락 비교
    "这家蛋糕太甜了，齁得我头疼。",        # FOOD → 부정 기대
    "她的笑容甜死人了，真温暖。",          # BEHAVIOR → 긍정 기대
]
print("\n[학습 전(Base) 추론]")
for s, p in zip(probe_samples, predict_proba(base_model_eval, probe_samples)):
    print(f"{s}\n  neg:{p[0]:.4f}  pos:{p[1]:.4f}")

In [ ]:
# ---------- 9) 모델 로드(학습용) ----------
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    torch_dtype=torch.float32
).to(device)

In [ ]:
# ---------- 10) 평가지표 ----------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(labels, probs[:,1])
    except:
        auc = 0.0
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1, "auc": auc}

In [ ]:
# ---------- 11) 학습 설정 & 풀 파인튜닝 ----------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("학습 및 저장 완료:", OUTPUT_DIR)

In [ ]:
# ---------- 12) 학습 후 점검(동일 샘플) ----------
fine_tuned = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR, num_labels=2).to(device)
fine_tuned.eval()

print("\n[학습 후(Full FT) 추론]")
for s, p in zip(probe_samples, predict_proba(fine_tuned, probe_samples)):
    print(f"{s}\n  neg:{p[0]:.4f}  pos:{p[1]:.4f}")

In [ ]:
# ---------- 13) 테스트셋 성능 비교(간단) ----------
def evaluate_model(model_eval, df_eval):
    texts = df_eval[TEXT_COL].astype(str).tolist()
    labels = df_eval["label_final"].astype(int).to_numpy()
    probs = []
    bs = 128
    for i in range(0, len(texts), bs):
        batch = texts[i:i+bs]
        probs.extend(predict_proba(model_eval, batch))
    probs = np.array(probs)
    preds = probs.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(labels, probs[:,1])
    except:
        auc = 0.0
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1, "auc": auc}

print("\n[테스트셋 성능]")
base_metrics = evaluate_model(base_model_eval, test_df)
ft_metrics   = evaluate_model(fine_tuned,      test_df)
print("Base :", base_metrics)
print("FullFT:", ft_metrics)

In [ ]:
# ---------- 14) 사용 가이드 ----------
print("""
[사용 Tip]
- 교정 로직은 4단계에서 정의됨:
  * "死了" + {可爱, 开心, ... , 乖} → 무조건 긍정(1)
  * '甜'은 domain/category가 있으면 그 값을 사용,
    없으면 음식 휴리스틱(FOOD_HINTS)에 따라 FOOD→부정(0), BEHAVIOR→긍정(1)
- 저장 경로: /content/erlangshen_fullfinetune_sile_fix
  추론 시: AutoModelForSequenceClassification.from_pretrained(해당 경로)
""")